# Metagenomic Assembly, Binning & MAGs

**Tier 3 — Applied Bioinformatics | Module 26 · Notebook 3**

*Prerequisites: Notebook 1 (Taxonomic Profiling), Module 17 (Genome Assembly)*

---

**By the end of this notebook you will be able to:**
1. Assemble a metagenome with MEGAHIT
2. Map reads back to contigs to estimate coverage (for binning)
3. Bin contigs into metagenome-assembled genomes (MAGs) with MetaBAT2
4. Assess MAG quality with CheckM (completeness and contamination)
5. Annotate high-quality MAGs with Prokka


**Key resources:**
- [MEGAHIT documentation](https://github.com/voutcn/megahit)
- [MetaBAT2 documentation](https://bitbucket.org/berkeleylab/metabat)
- [CheckM documentation](https://github.com/Ecogenomics/CheckM)

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This workflow maps to production-style applied bioinformatics: pipeline design, model evaluation, and biological interpretation for decision-making.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Pipeline outputs are context-dependent: QC failures upstream can invalidate downstream interpretation.
- Batch effects and confounders can dominate signal if not controlled explicitly.
- Use uncertainty-aware decisions: rank candidates and keep rationale for every threshold.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

# CSV preview (DNA)
with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

# Variants preview
with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

# FASTA preview
print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

# Metadata preview
meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

[← Previous: Functional Annotation](./02_functional_annotation.ipynb) | [Next: QIIME2 16S →](./04_qiime2_16s.ipynb)

---

## 1. Metagenomic Assembly with MEGAHIT

> Assemble decontaminated reads with MEGAHIT. Discuss k-mer size selection. Assess contig statistics (N50, # contigs > 1kb, max length).

In [ ]:
# Example: Assemble metagenome
# !megahit -1 decontam_1.fastq.gz -2 decontam_2.fastq.gz \
#     -o megahit_assembly/ --min-contig-len 500 -t 8

## 2. Coverage Estimation for Binning

> Map reads back to assembled contigs with Bowtie2. Calculate per-contig coverage and GC content (needed by MetaBAT2 for binning signal).

In [ ]:
# Example: Map reads back to assembly
# !bowtie2-build megahit_assembly/final.contigs.fa contigs_index
# !bowtie2 -x contigs_index -1 decontam_1.fastq.gz -2 decontam_2.fastq.gz | \
#     samtools sort -o contigs_mapped.bam
# !samtools index contigs_mapped.bam

## 3. Contig Binning with MetaBAT2

> Generate depth file from BAM. Run MetaBAT2 to cluster contigs into bins (MAGs). Report number of bins produced.

In [ ]:
# Example: MetaBAT2 binning
# !jgi_summarize_bam_contig_depths --outputDepth depths.txt contigs_mapped.bam
# !metabat2 -i megahit_assembly/final.contigs.fa -a depths.txt -o bins/bin --minContig 1500

## 4. MAG Quality with CheckM

> Run CheckM on bins. Interpret completeness (≥70-90%) and contamination (<5-10%) for medium/high quality MAGs (MIMAG standards).

In [ ]:
# Example: CheckM quality assessment
# !checkm lineage_wf bins/ checkm_out/ -t 8 -x fa
# !checkm qa checkm_out/lineage.ms checkm_out/ -o 2 > checkm_summary.txt

import pandas as pd
# Example: Load and filter high-quality MAGs
# checkm = pd.read_csv('checkm_summary.txt', sep='\t')
# hq_mags = checkm[(checkm['Completeness'] >= 90) & (checkm['Contamination'] < 5)]

## 5. MAG Annotation with Prokka

> Annotate a high-quality MAG with Prokka. Explore the GFF and protein FASTA output. Identify key metabolic genes.

In [ ]:
# Example: Annotate best MAG with Prokka
# !prokka bins/bin.1.fa --outdir prokka_bin1/ --prefix bin1 \
#     --metagenome --cpus 8

## Summary

> Summarize the assembly-binning-annotation pipeline. Discuss MAG taxonomy assignment (GTDB-Tk). Compare culture-based vs culture-independent genomics.

---

[← Previous: Functional Annotation](./02_functional_annotation.ipynb) | [Next: QIIME2 16S →](./04_qiime2_16s.ipynb)

---